# Stage 2: Exploratory Data Analysis — UrbanSound8K

This notebook covers:
- Class distribution for our 5 target classes
- Clip duration analysis
- Sample-rate distribution
- Waveform and spectrogram visualisation per class

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from config import (
    METADATA_CSV, AUDIO_DIR, TARGET_CLASS_IDS, TARGET_LABELS,
    SAMPLE_RATE, CLIP_DURATION, N_MELS, HOP_LENGTH, N_FFT,
)
from data_utils import load_metadata, load_waveform

plt.rcParams.update({
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor':   '#0d0d1a',
    'text.color':       'white',
    'axes.labelcolor':  '#aaa',
    'xtick.color':      '#aaa',
    'ytick.color':      '#aaa',
    'axes.edgecolor':   '#333',
    'grid.color':       '#222',
})
print('Setup complete!')

## 1 · Load Metadata

In [ ]:
df = load_metadata(filter_classes=True)
print(f'Total clips (5 classes): {len(df)}')
df.head()

## 2 · Class Balance

In [ ]:
class_counts = df['class'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
palette = sns.color_palette('husl', len(class_counts))
bars = ax1.bar(class_counts.index, class_counts.values, color=palette, edgecolor='#333')
for bar in bars:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(int(bar.get_height())), ha='center', color='white', fontsize=10)
ax1.set_title('Class Distribution (Bar)', fontsize=13, fontweight='bold', color='white')
ax1.set_xlabel('Class')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=20)

# Pie chart
ax2.pie(class_counts.values, labels=class_counts.index,
        autopct='%1.1f%%', colors=palette,
        textprops={'color': 'white'})
ax2.set_title('Class Distribution (Pie)', fontsize=13, fontweight='bold', color='white')

plt.suptitle('UrbanSound8K — Target Class Balance', fontsize=15, fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/plots/eda_class_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print(class_counts)

## 3 · Clip Duration Analysis

In [ ]:
from tqdm.notebook import tqdm

durations = []
sample_rates = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Reading metadata'):
    try:
        info = librosa.get_samplerate(row['filepath'])
        dur  = librosa.get_duration(path=row['filepath'])
        durations.append(dur)
        sample_rates.append(info)
    except:
        durations.append(None)
        sample_rates.append(None)

df['duration']    = durations
df['orig_sr']     = sample_rates

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Duration histogram
axes[0].hist(df['duration'].dropna(), bins=30, color='#00d4ff', edgecolor='#333')
axes[0].axvline(CLIP_DURATION, color='#ff6b6b', lw=2, linestyle='--', label=f'Cutoff ({CLIP_DURATION}s)')
axes[0].set_title('Clip Duration Distribution', fontweight='bold', color='white')
axes[0].set_xlabel('Duration (s)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Duration by class
df.boxplot(column='duration', by='class', ax=axes[1],
           boxprops=dict(color='#00d4ff'),
           medianprops=dict(color='#ff6b6b'),
           whiskerprops=dict(color='#aaa'),
           capprops=dict(color='#aaa'))
axes[1].set_title('Duration by Class', fontweight='bold', color='white')
axes[1].set_xlabel('Class')
axes[1].tick_params(axis='x', rotation=20)
plt.sca(axes[1])
plt.title('Duration by Class', color='white')

# Sample rate distribution
sr_counts = df['orig_sr'].value_counts()
axes[2].bar(sr_counts.index.astype(str), sr_counts.values,
            color='#7c3aed', edgecolor='#333')
axes[2].set_title('Original Sample Rates', fontweight='bold', color='white')
axes[2].set_xlabel('Sample Rate (Hz)')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=20)

plt.suptitle('Clip Duration & Sample Rate Analysis', fontsize=14, fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/plots/eda_duration.png', dpi=150, bbox_inches='tight')
plt.show()

print(df['duration'].describe())

## 4 · Waveform + Spectrogram Gallery (one per class)

In [ ]:
from features import extract_logmel

fig, axes = plt.subplots(5, 3, figsize=(18, 20))
classes = df['class'].unique()

for row_i, cls in enumerate(sorted(classes)):
    sample = df[df['class'] == cls].iloc[0]
    wave   = load_waveform(sample['filepath'])
    logmel = extract_logmel(wave)
    mfcc   = librosa.feature.mfcc(y=wave, sr=SAMPLE_RATE, n_mfcc=40)

    t = np.linspace(0, len(wave) / SAMPLE_RATE, len(wave))

    # Waveform
    axes[row_i, 0].plot(t, wave, color='#00d4ff', lw=0.6)
    axes[row_i, 0].set_title(f'{cls} — Waveform', color='white', fontsize=11)
    axes[row_i, 0].set_ylabel('Amplitude')

    # Log-Mel
    axes[row_i, 1].imshow(logmel, aspect='auto', origin='lower', cmap='magma')
    axes[row_i, 1].set_title(f'{cls} — Log-Mel', color='white', fontsize=11)

    # MFCC
    axes[row_i, 2].imshow(mfcc, aspect='auto', origin='lower', cmap='coolwarm')
    axes[row_i, 2].set_title(f'{cls} — MFCC', color='white', fontsize=11)

plt.suptitle('Audio Features — One Sample per Class',
             fontsize=16, fontweight='bold', color='white', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/plots/eda_spectrogram_gallery.png', dpi=120, bbox_inches='tight')
plt.show()
print('EDA complete!')